This project follows to find the best possible suitor for the next world cup in 2038

In [101]:
import pandas as pd
import numpy as np
import requests
import io
from sklearn.model_selection import train_test_split

headers = {
    "User-Agent": "Mozilla/5.0"
}

First data source: 2026_FIFA_World_Cup_squads from Wikipedia, using because it lists important data about the players such as: 

* official appearances a player has made for the national team
* for what club they play
* age
* name
* position

Scraped using this tutorial to make sure its actually legal to pull: [GeeksforGeeks](https://www.geeksforgeeks.org/python/web-scraping-from-wikipedia-using-python-a-complete-guide/) but using pandas instead of beautiful soup

In [102]:
wiki_url = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_squads"

wiki_html = requests.get(wiki_url, headers = headers)
wiki_df = pd.read_html(io.StringIO(wiki_html.text))

team_tables_df = []

for wiki_section in wiki_df:
    if 'Player' in wiki_section.columns:
        wiki_section = wiki_section.dropna(subset=['Player'])
        team_tables_df.append(wiki_section)

players_df = pd.concat(team_tables_df, ignore_index=True)

# cleanup player dataframe column names
players_df = players_df.drop(columns=["No."])
players_df = players_df.rename(columns = {"Date of birth (age)" : "Age", "Pos." : "Position", "Caps" : "Number of Appereances"})

# extract only the age of the player not the day they were born
players_df["Age"] = players_df["Age"].str.split()
players_df["Age"] = players_df["Age"].str[4]
players_df["Age"] = players_df["Age"].str.strip(')')

# take out the (captain) tag in the player name
players_df["Player"] = players_df["Player"].str.split('(')
players_df["Player"] = players_df["Player"].str[0]

# make sure all number columns are integers
players_df["Age"] = players_df["Age"].astype(int)
players_df["Number of Appereances"] = players_df["Number of Appereances"].astype(int)
players_df["Goals"] = players_df["Goals"].astype(int)

# put Player name as the first column
player_col = players_df.pop("Player")
players_df.insert(0, "Player", player_col)

players_df


,Player,Position,Age,Number of Appereances,Goals,Club
0,Matěj Kovář,GK,26,19,0,PSV Eindhoven
1,Jindřich Staněk,GK,30,14,0,Slavia Prague
2,Lukáš Horníček,GK,23,0,0,Braga
3,Vladimír Coufal,DF,33,61,2,TSG Hoffenheim
4,Tomáš Holeš,DF,33,39,2,Slavia Prague
...,...,...,...,...,...,...
1199,Azarias Londoño,MF,24,10,0,Universidad Católica
1200,José Fajardo,FW,32,65,17,Universidad Católica
1201,Ismael Díaz,FW,29,54,17,León
1202,Cecilio Waterman,FW,35,52,14,Universidad de Concepción


To fill in more information about our players we use an exisiting [Kaggle Dataset](https://www.kaggle.com/datasets/davidcariboo/player-scores?resource=download&select=clubs.csv). that pulls data from [Transfermarkt](https://www.transfermarkt.com/) . From the data set we extract the players.csv file. This will helps us include the following columns in our dataframe:
* country players were born in (previously very hard to do with wikipedia parsing)
* current market valuation in euros (most important number because it sets up what our regression tries to predict)
* peak market valuation in euros

In [103]:

# prepare the new columns
players_df['Club League'] = None
players_df['Country'] = None
players_df['Current MV'] = None
players_df['Peak MV'] = None

kaggle_df = pd.read_csv('./players.csv')

player_name = ""
for row_index, player_name in players_df['Player'].items():
    if player_name in kaggle_df['name'].values:
        kaggle_row = kaggle_df[kaggle_df['name'] == player_name].iloc[0]
        players_df.at[row_index, 'Club League'] = kaggle_row['current_club_domestic_competition_id']
        players_df.at[row_index, 'Country'] = kaggle_row['country_of_citizenship']
        players_df.at[row_index, 'Current MV'] = kaggle_row['market_value_in_eur']
        players_df.at[row_index, 'Peak MV'] = kaggle_row['highest_market_value_in_eur']

# normalize country names to be the same name (for old countries that no longer exist)
players_df['Country'] = players_df['Country'].replace({'CSSR': 'Czech Republic'})
players_df['Country'] = players_df['Country'].replace({'Jugoslawien (SFR)': 'Croatia'})

players_df


,Player,Position,Age,Number of Appereances,Goals,Club,Club League,Country,Current MV,Peak MV
0,Matěj Kovář,GK,26,19,0,PSV Eindhoven,None,None,None,None
1,Jindřich Staněk,GK,30,14,0,Slavia Prague,None,None,None,None
2,Lukáš Horníček,GK,23,0,0,Braga,None,None,None,None
3,Vladimír Coufal,DF,33,61,2,TSG Hoffenheim,L1,Czech Republic,2700000.0,12000000.0
4,Tomáš Holeš,DF,33,39,2,Slavia Prague,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...
1199,Azarias Londoño,MF,24,10,0,Universidad Católica,NaN,Panama,1200000.0,1200000.0
1200,José Fajardo,FW,32,65,17,Universidad Católica,None,None,None,None
1201,Ismael Díaz,FW,29,54,17,León,MEX1,Panama,1800000.0,2000000.0
1202,Cecilio Waterman,FW,35,52,14,Universidad de Concepción,None,None,None,None


Check number of missing values before and after stripping them off. As we can see the wikipedia dataset had all values introduced and none missing while the same cannot be said about the dataset from kaggle.

In [104]:
print('Before dropping missing values:')
print(players_df.isna().sum())
print('')

# drop values wherever there are missing values for the players for: 'Country', 'Current MV', 'Peak MV'
players_df = players_df.dropna(subset=['Country', 'Current MV', 'Peak MV', 'Club League']).reset_index(drop=True)

print('After dropping missing values:')
print(players_df.isna().sum())

Before dropping missing values:
Player                     0
Position                   0
Age                        0
Number of Appereances      0
Goals                      0
Club                       0
Club League              391
Country                  267
Current MV               283
Peak MV                  283
dtype: int64

After dropping missing values:
Player                   0
Position                 0
Age                      0
Number of Appereances    0
Goals                    0
Club                     0
Club League              0
Country                  0
Current MV               0
Peak MV                  0
dtype: int64


Finish normalizing columns by adding proper categories to variables/columns. We use 4 main types of variables types: object, categroy, int64 and bool.

In [ ]:
# make sure all number columns are integers
players_df["Current MV"] = players_df["Current MV"].astype(int)
players_df["Peak MV"] = players_df["Peak MV"].astype(int)

# fix column types to add the category type where needed
players_df['Position'] = players_df['Position'].astype('category')
players_df['Club'] = players_df['Club'].astype('category')
players_df['Club League'] = players_df['Club League'].astype('category')
players_df['Country'] = players_df['Country'].astype('category')

# based on the current valuation and peak valuation add a bool type column if they are in the peak of their career
players_df['Apex MV'] = (players_df['Current MV'] == players_df['Peak MV']).astype('bool')

print(players_df.dtypes)

Player                     object
Position                 category
Age                         int64
Number of Appereances       int64
Goals                       int64
Club                     category
Club League              category
Country                  category
Current MV                  int64
Peak MV                     int64
Apex MV                      bool
dtype: object


Split dataset into a training and testing set. Snippet de cod luat din laborator. Reserve 75% of data for training and 25% for testing. Export created dataset into the train.csv and test.csv to help run the models later.

In [106]:
target = players_df['Current MV']
features = players_df.drop(columns=['Current MV', 'Player'])

target.to_csv('test.csv')
features.to_csv('train.csv')

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, random_state=42)

test_data_export = pd.concat([X_test, y_test], axis=1)
train_data_export = pd.concat([X_train, y_train], axis=1)

test_data_export.to_csv('test.csv', index=False)
train_data_export.to_csv('train.csv', index=False)

print(f"Test set data dimension: {test_data_export.shape}")
print(f"Training set data dimension: {train_data_export.shape}")

Test set data dimension: (200, 10)
Training set data dimension: (597, 10)
